# Data ETL and exploration/preprocessing

## Loading and Exploring Data

### Guidelines

1. Load all the kcmetro text data independently and drop irrelevant information
2. Merge all data 
3. Explore data and calendar/caldates/service id in particular
4. Create function to calculate frequencies from schedule data
5. create function to map routes that match frequency target 

Libraries to Use:

-numpy and pandas 
-geopandas and folium
-datetime 
-os and shutils 
-arcpy

In [89]:
###installs

%pip install folium pandas numpy geopandas requests

Note: you may need to restart the kernel to use updated packages.


In [90]:
#imports
import numpy as np, pandas as pd, requests, json, os, zipfile
from pathlib import Path
import folium as fl, geopandas as gp 
from datetime import datetime



In [91]:
gtfs_url = r'https://gtfs.sound.obaweb.org/prod/gtfs_puget_sound_consolidated.zip'
output_dir = '../data/pgc_gtfs'

def download_and_unzip(url, out_dir):
    zip_dir = output_dir + '/gtfs_consolidated.zip'
    response = requests.get(url)
    with open (zip_dir,'wb') as f:
        f.write(response.content)

    with zipfile.ZipFile(zip_dir, 'r') as zip_ref:
        zip_ref.extractall(output_dir)

download_and_unzip(gtfs_url, output_dir)

In [92]:
###loading data
trips = pd.read_csv(r'../data/pgc_gtfs/trips.txt')
stop_times = pd.read_csv(r'../data/pgc_gtfs/stop_times.txt', dtype={'trip_id': str, 'stop_id': str, 'stop_headsign': str})
stops = pd.read_csv(r'../data/pgc_gtfs/stops.txt')
routes = pd.read_csv(r'../data/pgc_gtfs/routes.txt')
cal = pd.read_csv(r'../data/pgc_gtfs/calendar.txt')
cal_dates = pd.read_csv(r'../data/pgc_gtfs/calendar_dates.txt')

In [93]:
###dropping unnecessary information

trips_clean = trips[['route_id','service_id','trip_id','trip_headsign', 'direction_id','block_id','shape_id']]
stop_times_clean = stop_times[['trip_id','arrival_time','departure_time','stop_id','stop_sequence','shape_dist_traveled','timepoint']]
stops_clean = stops[['stop_id','stop_code','stop_name','stop_lat','stop_lon']]
routes_clean = routes[['route_id','agency_id','route_short_name','route_desc','route_type','route_color','route_text_color']]

###cal and calendar dates are to remain unchanged until I explore the data more thoroughly 
###set the variables to the clean dataframes

trips = trips_clean
stop_times = stop_times_clean
stops = stops_clean
routes = routes_clean


###find and set types
stop_times.arrival_time = pd.to_timedelta(stop_times.arrival_time)
stop_times.departure_time = pd.to_timedelta(stop_times.departure_time)



In [94]:
###merge together data

#merge all files one by one 

data_merged = pd.merge(trips, stop_times, how='left', on='trip_id')
data_merged = pd.merge(data_merged, stops, how='left', on='stop_id')
data_merged = pd.merge(data_merged, routes, how='left', on='route_id')

###filter to only kcm bus routes 
data_merged = data_merged.loc[data_merged.agency_id.isin([1])]

data_merged.agency_id.unique()

array([1])

### Data Exploration

In [95]:
###let us explore the service id to see which routes have which ids

route_servs = pd.DataFrame(data_merged.groupby('route_short_name').service_id.unique())
route_servs['unique_service_ids'] = route_servs.service_id.apply(lambda x: x.shape[0])

###put unique service ids in the front of the dataframe 
route_servs = route_servs.iloc[:,[1,0]]

data_merged.service_id.value_counts()

service_id
130540          416055
25266           321905
25530           304471
2295             20862
128245           20474
22239             7242
3291              5764
63424.171903        96
110639              87
21836               71
87923               25
108258              10
Name: count, dtype: int64

As we can see, the most popular service ids are 130540, 25266, and 25530 - corresponding to weekday,sat, and sun service

In [96]:
####find calendar dates to confirm my intuition 
cal_serv = cal

###add total weekday count and set to one for boolean comparisons 
cal_serv['weekday'] = cal_serv.drop(columns=['service_id', 'start_date','end_date', 'saturday','sunday']).apply(lambda x: any(x), axis=1)

####drop individual day of the workweek columns 
cal_serv = cal_serv.drop(columns=cal_serv.columns[1:6])

### us DF.dot to multiply 
cal_serv['dow'] = cal_serv[['saturday','sunday','weekday']].dot(cal_serv.columns[[1,2,5]] + '_').str.lower().apply(lambda x: x.strip('_'))

###find services that are weekday, weekend, or special service
cal_serv = cal_serv.drop(columns=['saturday','sunday','weekday'])
cal_serv.dow = cal_serv.dow.replace('', 'special')

###make service id an str
cal_serv.service_id = cal_serv.service_id.astype(str)

###save service ids by dow for further exploration
cal_serv.to_csv('../data/serv_dow.csv')

###display 
cal_serv.dow.unique()


<ArrowStringArray>
['special', 'weekday', 'sunday', 'saturday', 'sunday_weekday',
 'saturday_sunday']
Length: 6, dtype: str

### Frequency Calculations/Functions and Verification

We will explore the data by using various methods to try to calculate frequency and then we will validate. 

In [97]:
##lets use groupby in pandas to filter/sort the data we desire and limit it to a single stop + dir + service_id 
data_merged = data_merged.sort_values(by=['stop_id', 'arrival_time'])

####group by stop id and sort by time of arrival 
data_merged['prev_depart'] = data_merged.groupby(['route_id', 'service_id', 'direction_id', 'stop_id']).departure_time.shift(1)
data_merged['gap'] = data_merged['departure_time'] - data_merged['prev_depart']
route_freq = pd.DataFrame(data_merged.groupby('route_short_name').gap.mean().sort_values(ascending=True))

##group by stop sequence and sort by time of arrival
data_merged['prev_depart'] = data_merged.groupby(['route_id', 'service_id', 'direction_id', 'stop_sequence']).departure_time.shift(1)
data_merged['gap'] = data_merged['departure_time'] - data_merged['prev_depart']
route_freq['by_sequence'] = pd.DataFrame(data_merged.groupby('route_short_name').gap.mean().sort_values(ascending=True))


### we will now find the differences between sorting by stop sequence and sorting by stop_id 
route_freq['dif'] = np.abs(route_freq['by_sequence'] - route_freq['gap'])

###display the differences based on calculation type. 
###it would seem sensible to me that stop id would be the better variable to use as lines can start at various stop ids first (changing stop seq no.)
route_freq.loc[route_freq.dif >= pd.Timedelta(seconds=1)].sort_values('dif')['dif']

route_short_name
107              0 days 00:00:07.554677
161              0 days 00:00:08.477619
A Line           0 days 00:00:09.365427
E Line           0 days 00:00:10.490623
49               0 days 00:00:11.466016
245              0 days 00:00:15.483792
215              0 days 00:00:19.084447
70               0 days 00:00:19.730650
226              0 days 00:00:25.181473
164              0 days 00:00:25.711766
125              0 days 00:00:27.236255
914              0 days 00:00:27.726881
223              0 days 00:00:34.156468
62               0 days 00:00:37.606682
27               0 days 00:00:46.029827
C Line           0 days 00:00:50.230125
132              0 days 00:01:00.403184
220              0 days 00:01:04.092731
204              0 days 00:01:06.470390
106              0 days 00:01:08.027095
348              0 days 00:01:13.306920
102              0 days 00:01:22.363254
131              0 days 00:01:30.489484
150              0 days 00:01:32.961136
775              0 days

From here we will validate data using a few routes and accurate schedule data

In [98]:
###find data from rapidrides to compare


rr_data = data_merged.loc[data_merged.route_short_name.str.match(r'\w Line')]
rr_freq = route_freq.loc[route_freq.index.str.match(r'\w Line')]


In [99]:
###function for calculating frequencies based on a time range and a label (departure_time or arrival_time)
def freq_calc(df, label='departure_time', start_hr=0, end_hr=30):

    ### only locate columns we need for the calculations to save memory and increase run speed
    df_lite = df[['route_id','trip_id','stop_id','route_short_name', 'service_id','direction_id', label]]

    start_time = pd.Timedelta(hours=start_hr)
    end_time = pd.Timedelta(hours=end_hr)
    ###convert departure_time to timedelta
    df_lite.departure_time = pd.to_timedelta(df_lite.departure_time)
    ###restrict to time range
    df_lite = df_lite.loc[(df_lite.departure_time >= start_time) & (df_lite.departure_time <= end_time)]

    ####group by stop id and sort by time of arrival 
    df_lite['prev_depart'] = df_lite.groupby(['route_id', 'service_id', 'direction_id', 'stop_id']).departure_time.shift(1)
    df_lite['gap'] = df_lite['departure_time'] - df_lite['prev_depart']

    ###calculate frequencies 
    route_freq = pd.DataFrame(df_lite.groupby('route_short_name').gap.mean())

    route_freq['mean_gap'] = route_freq['gap']
    route_freq = route_freq.drop(columns='gap')

    route_freq['median_gap'] = df_lite.groupby('route_short_name').gap.median()

    return route_freq


    

In [100]:
###function for calculating first and last arrivals 

def first_last(df, label='departure_time', start_hr=0, end_hr=30):

    ### only locate columns we need for the calculations to save memory and increase run speed
    df_lite = df[['route_id','trip_id','stop_id','route_short_name', 'service_id','direction_id', label]]

    start_time = pd.Timedelta(hours=start_hr)
    end_time = pd.Timedelta(hours=end_hr)
    ###convert departure_time to timedelta
    df_lite.departure_time = pd.to_timedelta(df_lite.departure_time)
    ###restrict to time range
    df_lite = df_lite.loc[(df_lite.departure_time >= start_time) & (df_lite.departure_time <= end_time)]

    ####find first and last departure times for each route
    fl_df = df_lite.groupby('route_short_name')[label].agg(['min', 'max']).rename(columns={'min': 'first_departure', 'max': 'last_departure'})

    return fl_df

In [101]:
###calculate frequencies for each day of the week and time range and save to csv files for later use in mapping/streamlit dashboard
data_merged

for dow in ['weekday','saturday','sunday']:
    for times in [(0,30),(6,9),(9,15),(15,19),(19,24),(20,30)]:
        start_hr = times[0]
        end_hr = times[1]
        print(f'Calculating frequencies for {dow} from {start_hr} to {end_hr}')

        serv_ids = cal_serv.loc[cal_serv.dow == dow].service_id.unique()
        data_dow = data_merged.loc[data_merged.service_id.isin(serv_ids)]

        freq_df = freq_calc(data_dow, label='departure_time', start_hr=start_hr, end_hr=end_hr)
        freq_df[['first','last']] = first_last(data_dow, label='departure_time', start_hr=start_hr, end_hr=end_hr)
        freq_df['trip_count'] = data_dow.groupby('route_short_name').trip_id.nunique()

        freq_df.to_csv(f'../data/freq_data/freq_{dow}_{start_hr}_{end_hr}.csv')




Calculating frequencies for weekday from 0 to 30
Calculating frequencies for weekday from 6 to 9
Calculating frequencies for weekday from 9 to 15
Calculating frequencies for weekday from 15 to 19
Calculating frequencies for weekday from 19 to 24
Calculating frequencies for weekday from 20 to 30
Calculating frequencies for saturday from 0 to 30
Calculating frequencies for saturday from 6 to 9
Calculating frequencies for saturday from 9 to 15
Calculating frequencies for saturday from 15 to 19
Calculating frequencies for saturday from 19 to 24
Calculating frequencies for saturday from 20 to 30
Calculating frequencies for sunday from 0 to 30
Calculating frequencies for sunday from 6 to 9
Calculating frequencies for sunday from 9 to 15
Calculating frequencies for sunday from 15 to 19
Calculating frequencies for sunday from 19 to 24
Calculating frequencies for sunday from 20 to 30
